# Module 02 — Dask

**Formation Big Data — ANSD / Data Innovation Lab**

Troisième outil du bloc, et changement d'objectif. Polars et DuckDB cherchent à
tirer le maximum d'**une** machine. Dask répond à une autre question : comment
faire tourner du code de type pandas sur des données découpées en morceaux, et
comment passer de votre portable à **plusieurs machines** sans réécrire le code.

> **À lire avant de commencer.** Sur le volume de ce TP et sur une seule
> machine, Dask sera vraisemblablement **plus lent** que Polars et DuckDB, et
> peut-être plus lent que pandas. Ce n'est pas un défaut de configuration :
> découper le travail et coordonner les morceaux a un coût, qui n'est amorti
> qu'à partir d'un certain volume — ou lorsque plusieurs machines entrent en
> jeu. Nous le mesurerons, et nous en tirerons la conclusion utile.
>
> L'intérêt de ce notebook n'est donc pas la performance, mais le **modèle
> d'exécution** : partitions, graphe de tâches, exécution différée. Ces trois
> notions sont exactement celles de Spark, que nous verrons plus tard. Ce que
> vous comprenez ici, vous n'aurez pas à le réapprendre.

Au programme :

1. démarrer un cluster local et ouvrir son **tableau de bord** ;
2. découvrir les **partitions** ;
3. comprendre l'exécution différée et `.compute()` ;
4. lire un plan d'exécution et y reconnaître le map / reduce ;
5. `map_partitions` : appliquer du pandas à chaque morceau ;
6. régler le nombre de partitions — un vrai savoir-faire ;
7. la campagne de mesures.

## 1. Protocole et contexte

In [ ]:
%load_ext autoreload 
%autoreload 2
import sys
from pathlib import Path
try:
    sys.path.append(str(Path(__file__).parent.parent.resolve()))
except NameError:
    sys.path.append(str(Path.cwd().parent.resolve()))
from tools.outils_mesure import (FICHIER, FICHIER_REGIONS, VOLUME_COMPARAISON,
                           afficher_protocole, contexte_machine, enregistrer,
                           memoire_mo, mesurer)
import re
import warnings
warnings.filterwarnings("ignore")
import dask
import dask.dataframe as dd

print("Dask", dask.__version__)
print()
contexte_machine()

## 2. Démarrer un cluster local

Un « cluster » sur votre propre machine : plusieurs processus de travail, un
ordonnanceur qui répartit les tâches, et un tableau de bord pour observer le
tout. C'est la même architecture que sur un vrai cluster de plusieurs serveurs
— seule l'échelle change.

In [ ]:
from dask.distributed import Client, LocalCluster

# Par défaut, Dask choisit le nombre de processus selon vos cœurs.
# Si votre machine est juste en mémoire, réduisez : n_workers=2, memory_limit="1GB"
cluster = LocalCluster()
client = Client(cluster)

print("Processus de travail :", len(client.scheduler_info()["workers"]))
print("Tableau de bord      :", client.dashboard_link)
client

### 👉 Ouvrez le tableau de bord maintenant

Cliquez sur le lien ci-dessus, ou copiez-le dans un onglet de votre navigateur.
Disposez-le à côté de ce notebook : nous allons le regarder pendant chaque
calcul.

Trois onglets à connaître :

- **Status** — les tâches en cours, l'occupation des processeurs, la mémoire
- **Workers** — l'état de chaque processus de travail
- **Graph** — le graphe des tâches, en cours d'exécution

C'est la première fois de la semaine que vous allez **voir** le parallélisme.

## 3. Premier contact : ce n'est pas un DataFrame

`dd.read_csv` ressemble à `pd.read_csv`, mais ne lit presque rien : il repère
le fichier, en déduit les colonnes et leurs types sur un échantillon, et prépare
un découpage.

In [ ]:
ddf = dd.read_csv(FICHIER, blocksize="64MB")
ddf

Observez l'affichage : aucune donnée, seulement la structure et un nombre
de partitions. Dask vous montre le **plan de la table**, pas la table.

In [ ]:
print("Nombre de partitions :", ddf.npartitions)
print("Colonnes             :", len(ddf.columns))
print()
ddf.dtypes

> ⚠️ **Piège classique.** Dask devine les types sur un échantillon du début
> du fichier. Si une colonne contient des entiers dans les premières lignes et
> du texte plus loin, les partitions seront incohérentes et le calcul échouera
> en cours de route. Sur des données administratives, c'est fréquent. La parade
> est d'imposer les types : `dd.read_csv(..., dtype={"nb_pieces": "float64"})`.

### Les partitions : des DataFrames pandas empilés

Une table Dask **est** une collection de tables pandas. C'est la clé de tout le
reste.

In [ ]:
# Récupérer une partition, et une seule : c'est un vrai DataFrame pandas
partition = ddf.get_partition(0).compute()
print(type(partition))
partition.head(3)

In [ ]:
# Combien de lignes dans chaque partition ?
lignes_par_partition = ddf.map_partitions(len).compute()
print(lignes_par_partition.tolist())
print(f"\nTotal : {lignes_par_partition.sum():,} lignes".replace(",", " "))

**Question 1.** Les partitions ont-elles toutes exactement le même nombre
de lignes ? Pourquoi, sachant que le découpage se fait par taille en octets ?

*Votre réponse :* …

## 4. L'exécution différée

Voici le point qui déroute, et le plus important du notebook.

In [ ]:
moyenne = ddf["age"].mean()
print(type(moyenne))
moyenne

Vous avez demandé une moyenne, et vous avez reçu… une promesse. Rien n'a
été calculé. Pour obtenir la valeur, il faut la réclamer explicitement.

**Regardez le tableau de bord** pendant l'exécution de la cellule suivante.

In [ ]:
resultat, duree, _ = None, None, None
m = mesurer("moyenne d'âge", lambda: ddf["age"].mean().compute())

**Question 2.** Qu'avez-vous observé dans l'onglet *Status* du tableau de
bord ? Combien de barres se sont activées ? Comparez avec le notebook 02, où un
seul cœur travaillait.

*Votre réponse :* …

### Une erreur à ne pas commettre

`.compute()` ramène le résultat **en mémoire, dans votre notebook**. C'est
parfait pour une moyenne ou un tableau agrégé. C'est catastrophique sur une
table de plusieurs millions de lignes : vous retombez exactement sur le mur du
notebook 02.

In [ ]:
# Parmi ces trois expressions, laquelle fait revenir tout le fichier ?
#
#   A : ddf.groupby("region")["age"].mean().compute()   → sûre
#       Le résultat tient en quelques lignes : une par région.
#
#   B : ddf[ddf.age >= 15].compute()                    → DANGEREUSE
#       C'est un filtre, pas une agrégation : le résultat compte des
#       millions de lignes, toutes rapatriées dans ce notebook.
#
#   C : ddf["situation_activite"].value_counts().compute()  → sûre
#       Une ligne par modalité, soit six lignes.
#
# On exécute donc A et C, et l'on se contente de COMPTER pour B.

print("A — âge moyen par région :")
print(ddf.groupby("region")["age"].mean().compute().head())

print("\nC — effectifs par situation d'activité :")
print(ddf["situation_activite"].value_counts().compute())

print("\nB — on compte, sans rapatrier :")
print(f"{len(ddf[ddf.age >= 15]):,} personnes de 15 ans et plus"
      .replace(",", " "))

## 5. Le plan d'exécution

Comme Polars et DuckDB, Dask optimise avant d'exécuter. L'affichage natif est
très verbeux ; la fonction ci-dessous n'en garde que la structure.

In [ ]:
MOTIF_NOEUD = re.compile(r"^[\s|]*[A-Z]\w*[\(:]")


def plan(collection, largeur=70):
    """Affiche le plan d'exécution optimisé, sans le bruit."""
    for ligne in collection.optimize()._expr.tree_repr().splitlines():
        if MOTIF_NOEUD.match(ligne):
            print(ligne[:largeur] + ("…" if len(ligne) > largeur else ""))


plan(ddf[ddf.age > 15].groupby("region")["age"].mean())

Lisez ce plan de bas en haut, c'est l'ordre d'exécution :

- `FromMapProjectable` — la lecture du fichier, morceau par morceau
- `Projection: columns='age'` — **seule la colonne `age` sera lue**, comme chez
  Polars et DuckDB
- `Filter` / `GT` — le filtre sur l'âge
- `Mean(GroupByChunk)` — une moyenne partielle **dans chaque partition**
- `Mean(TreeReduce)` — les résultats partiels **recombinés**

**Question 3.** Les deux dernières étapes portent un nom qui devrait vous
rappeler quelque chose du début du bloc. Lequel, et pourquoi cette
décomposition en deux temps est-elle indispensable ?

*Votre réponse :* …

C'est le **map / reduce** de la slide sur le parallélisme, rendu visible :
chaque morceau est traité indépendamment, puis les résultats sont combinés.
C'est exactement ce que fait Spark, avec les mêmes mots.

## 6. `map_partitions` : du pandas, sur chaque morceau

Puisque chaque partition est un DataFrame pandas, on peut lui appliquer
n'importe quelle fonction pandas. C'est l'échappatoire universelle de Dask :
tout ce que vous savez faire en pandas, vous pouvez le faire ici.

In [ ]:
# Exemple fourni : une fonction pandas ordinaire, appliquée à chaque partition
def nettoyer(partition_pandas):
    partition_pandas = partition_pandas.copy()
    partition_pandas["region"] = (partition_pandas["region"]
                                  .str.strip()
                                  .str.title())
    return partition_pandas


ddf_propre = ddf.map_partitions(nettoyer)
ddf_propre["region"].nunique().compute()

In [ ]:
# Nombre d'âges aberrants, compté partition par partition.
#
# `map_partitions` applique une fonction pandas ordinaire à chaque morceau :
# chaque partition EST un DataFrame pandas.

def compter_aberrants(partition_pandas):
    return ((partition_pandas["age"] < 0) | (partition_pandas["age"] > 110)).sum()


par_partition = ddf.map_partitions(compter_aberrants).compute()

print("Par partition :", par_partition.tolist())
print(f"Total : {par_partition.sum()} âges impossibles")

**Question 4.** Cette approche a une limite. Que se passerait-il si votre
fonction avait besoin de **comparer des lignes situées dans des partitions
différentes** — par exemple, détecter des doublons sur tout le fichier ?

*Votre réponse :* …

C'est toute la difficulté du calcul distribué, et la raison pour laquelle les
tris et les jointures y sont coûteux : ils obligent à faire circuler des données
entre les partitions. Retenez le mot : on appelle cela un **brassage**
(*shuffle*). Il reviendra au jour 3.

## 7. `persist` ou `compute` ?

- `.compute()` — exécute et **ramène le résultat dans votre notebook**
- `.persist()` — exécute et **garde le résultat réparti dans le cluster**

`persist` est ce qu'on utilise quand on va réutiliser plusieurs fois la même
table : sans lui, Dask relit et recalcule tout à chaque `.compute()`.

In [ ]:
# Sans persist : chaque calcul relit le fichier
print("Sans persist :")
m1 = mesurer("1er calcul", lambda: ddf_propre["age"].mean().compute())
m2 = mesurer("2e calcul",  lambda: ddf_propre["age"].max().compute())

# Avec persist : la table reste en mémoire dans le cluster
ddf_persiste = ddf_propre.persist()
_ = ddf_persiste["age"].mean().compute()   # on attend la fin du chargement

print("\nAvec persist :")
m3 = mesurer("1er calcul", lambda: ddf_persiste["age"].mean().compute())
m4 = mesurer("2e calcul",  lambda: ddf_persiste["age"].max().compute())

**Question 5.** Quel est le gain ? Et quel est le risque de `persist` sur
une table qui dépasserait la mémoire du cluster ?

*Votre réponse :* …

## 8. Régler le nombre de partitions

C'est le réglage le plus important de Dask — et le même existera sur Spark.

- **Trop peu de partitions** : les processus de travail n'ont rien à faire, et
  chaque morceau est trop gros pour la mémoire
- **Trop de partitions** : le coût de coordination dépasse le gain

La règle usuelle : des partitions de l'ordre de 100 Mo, et au moins autant de
partitions que de cœurs disponibles.

In [ ]:
# Effet du découpage sur le temps de calcul
import time

for taille_bloc in ["16MB", "64MB", "256MB"]:
    d = dd.read_csv(FICHIER, blocksize=taille_bloc)
    print(f"blocksize={taille_bloc:>6} → {d.npartitions:>3} partitions", end="  ")
    depart = time.perf_counter()
    d.groupby("region")["age"].mean().compute()
    print(f"→ {time.perf_counter() - depart:5.1f} s")

**Question 6.** Quel découpage est le plus rapide sur votre machine ?
Combien de partitions cela représente-t-il par cœur ?

*Votre réponse :* …

## 9. Campagne de mesures

Les cinq opérations de référence, dans l'ordre du protocole.

> ⏳ **Cette campagne est longue** — plusieurs minutes, notamment sur le tri et
> la jointure, qui obligent Dask à faire circuler les données entre partitions.
> Gardez le tableau de bord ouvert : le brassage y est très visible. Cette
> lenteur fait partie de l'enseignement.

In [ ]:
# Fourni : préparation de la campagne
#
# Attention : on ne peut pas utiliser `.head()` pour se limiter à 2 millions de
# lignes — cela ramènerait tout dans UNE seule partition, et Dask perdrait
# précisément ce qu'on veut mesurer. On retient donc les premières partitions
# jusqu'à approcher le volume voulu.

ddf = dd.read_csv(FICHIER, blocksize="64MB")
lignes_par_partition = ddf.map_partitions(len).compute()
k = int((lignes_par_partition.cumsum() < VOLUME_COMPARAISON).sum()) + 1
k = min(k, ddf.npartitions)

ddf_mesure = ddf.partitions[:k].persist()
lignes_mesure = len(ddf_mesure)
reference = ddf_mesure[["id_individu", "nom"]].sample(frac=0.5, random_state=1).persist()

print(f"{k} partitions retenues → {lignes_mesure:,} lignes".replace(",", " "))
print("(le volume ne tombe pas exactement sur 2 millions : on ne coupe pas "
      "une partition en deux)")

In [ ]:
# Mesurez les cinq opérations, dans cet ordre exact.
# N'oubliez pas `.compute()` : sans lui, vous chronométrez la construction du
# graphe, pas le calcul.
#
#   lecture     : dd.read_csv(...).compute()  (ramène tout en mémoire)
#   filtre      : les 15 ans et plus
#   agregation  : âge moyen par région
#   tri         : tri par âge
#   jointure    : jointure de ddf_mesure avec reference sur id_individu
from dask.distributed import wait
dask.config.set({"distributed.scheduler.allowed-failures": 0})
# Sans cela, un worker tué est retenté 3 fois avant d'abandonner : la cellule
# semble bloquée alors qu'elle échoue. À 0, l'échec est immédiat et net.
# Si besoin, intercaler un client.restart() entre les mesures pour repartir sur un cluster propre.

mesures = []
mesures.append(mesurer("lecture",    lambda: ddf.partitions[:k].compute()))
mesures.append(mesurer("filtre",     lambda: wait(ddf_mesure[ddf_mesure.age >= 15].persist())))
mesures.append(mesurer("agregation", lambda: ddf_mesure.groupby("region")["age"].mean().compute()))
mesures.append(mesurer("tri",        lambda: wait(ddf_mesure.sort_values("age").persist())))
mesures.append(mesurer("jointure",   lambda: wait(ddf_mesure.merge(reference, on="id_individu", how="left").persist())))

enregistrer(mesures, outil="dask", volume="2M", lignes=lignes_mesure)

### Passe sur le fichier complet

In [ ]:
# Répétez les cinq mesures sur le fichier complet.
# Les échecs seront enregistrés comme tels : c'est un résultat.

del ddf_mesure, reference
import gc; gc.collect()

ddf_complet = dd.read_csv(FICHIER, blocksize="64MB")
reference_complete = ddf_complet[["id_individu", "nom"]].sample(frac=0.5, random_state=1)
lignes_completes = len(ddf_complet)
print(f"{lignes_completes:,} lignes".replace(",", " "))

mesures_completes = []
mesures_completes.append(mesurer("lecture",    lambda: ddf_complet.partitions[:k].compute()))
mesures_completes.append(mesurer("filtre",     lambda: wait(ddf_complet[ddf_complet.age >= 15].persist())))
mesures_completes.append(mesurer("agregation", lambda: ddf_complet.groupby("region")["age"].mean().compute()))
mesures_completes.append(mesurer("tri",        lambda: wait(ddf_complet.sort_values("age").persist())))
mesures_completes.append(mesurer("jointure",   lambda: wait(ddf_complet.merge(reference_complete, on="id_individu", how="left").persist())))

enregistrer(mesures_completes, outil="dask", volume="complet",
            lignes=lignes_completes)

In [ ]:
# Fermeture propre du cluster
client.close()
cluster.close()
print("Cluster arrêté.")

## 10. Ce qu'il faut retenir

- Une table Dask est une **collection de DataFrames pandas** : les partitions.
- Rien ne s'exécute avant `.compute()`. Ce que vous manipulez, c'est un **graphe
  de tâches**, pas des données.
- Le plan révèle le **map / reduce** : un calcul partiel par partition, puis une
  recombinaison.
- `map_partitions` permet d'appliquer n'importe quelle fonction pandas à chaque
  morceau — mais seulement si la fonction n'a pas besoin de voir les autres.
- Les tris et jointures exigent un **brassage** entre partitions : c'est ce qui
  coûte cher, ici comme sur Spark.
- Le **nombre de partitions** est un réglage à part entière.
- Sur une seule machine et à ce volume, Dask ne rivalise pas avec Polars ou
  DuckDB. Son domaine commence là où une machine ne suffit plus — et le même
  code passe alors sur plusieurs serveurs, sans réécriture.

**À compléter :**

- Processus de travail utilisés : … · partitions retenues : …
- Gain apporté par `persist` : ×…
- Opération la plus coûteuse : … (pourquoi ? …)
- Échecs sur le fichier complet : …
